# Conditional Routing & Routers in LangGraph

In traditional linear chains, every step runs in a fixed, predetermined order (`A -> B -> C`).
However, real-world agentic applications require **dynamic decision-making**:
- Inspecting the current graph state and deciding which specialized node should execute next.
- Routing user requests to domain experts (e.g., Math specialist vs. Coding specialist vs. General QA).
- Dynamically branching between executing external tools and returning a final answer.

In LangGraph, this is achieved through **Routers** created via `add_conditional_edges()`.

---

### 🎨 LangGraph Conditional Routing Architecture

![LangGraph Conditional Routing Architecture](../images/router_architecture.png)

---

### Key Concepts Covered in this Notebook:
1. **Core Anatomy of a Router**: Understanding `add_conditional_edges(source, router_function, path_map)`.
2. **Pattern 1: Deterministic Rule-Based Router**: Fast, zero-token cost keyword/state branching (e.g. Support ticket triage).
3. **Pattern 2: Semantic LLM Intent Router**: Using an LLM to classify user intent and route to specialized domain experts.
4. **Pattern 3: Agentic Tool-Calling Router (`tools_condition`)**: Dynamic branching between tool execution and conversation completion.
5. **Routing Strategy Matrix**: When to use deterministic vs. semantic vs. tool routers.

---
## 0. Setup & Free LLM Initialization

We load our environment variables and initialize **Groq** (`qwen/qwen3.8-27b`) as our fast, free LLM.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Add project root to sys.path so llm_config can be imported from any subdirectory
for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "llm_config.py").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from llm_config import get_llm, list_free_models

# Load environment variables (.env)
load_dotenv(find_dotenv())

# Initialize our free model with automatic multi-model fallback
free_model = get_llm(temperature=0.0)  # Zero temperature for deterministic routing classifications
print("✅ Model initialized successfully with automatic fallbacks")


---
## 1. Pattern 1: Deterministic Rule-Based Router

A **Deterministic Router** uses standard Python logic (keywords, state flags, thresholds) to route without calling an LLM.

**Advantages:**
- **Near-zero latency** (<1 ms)
- **Zero API cost** ($0.00)
- **100% predictable & reproducible**

### Example: Customer Support Triage
- Billing inquiries -> `billing_node`
- Technical issues -> `technical_node`
- Everything else -> `general_node`

In [2]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. Define State Schema
class SupportState(TypedDict):
    query: str
    category: str
    response: str

# 2. Triage Node (Categorizes the query using fast keyword heuristics)
def triage_node(state: SupportState):
    q = state["query"].lower()
    if any(w in q for w in ["refund", "billing", "invoice", "payment", "charge"]):
        cat = "billing"
    elif any(w in q for w in ["error", "crash", "bug", "timeout", "server", "504"]):
        cat = "technical"
    else:
        cat = "general"
    return {"category": cat}

# 3. Router Function (Returns the path key)
def route_support(state: SupportState) -> Literal["billing", "technical", "general"]:
    return state["category"]

# 4. Specialized Destination Nodes
def billing_node(state: SupportState):
    return {"response": f"[Billing Dept]: Processing refund/invoice request for '{state['query']}'."}

def technical_node(state: SupportState):
    return {"response": f"[Tech Support]: Investigating system logs for error in '{state['query']}'."}

def general_node(state: SupportState):
    return {"response": f"[Customer Care]: Assisting with inquiry: '{state['query']}'."}

# 5. Build and Compile the Graph
builder = StateGraph(SupportState)
builder.add_node("triage", triage_node)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_node("general", general_node)

builder.add_edge(START, "triage")

# Conditional edge: triage -> route_support -> path_map
builder.add_conditional_edges(
    "triage",
    route_support,
    {
        "billing": "billing",
        "technical": "technical",
        "general": "general",
    }
)

builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("general", END)

support_graph = builder.compile()

# Graph Visualization for support_graph
from IPython.display import Image, display
try:
    display(Image(support_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(support_graph.get_graph().draw_ascii())

# 6. Test with diverse queries
test_queries = [
    "I have a billing issue with my invoice.",
    "The API server returned a 504 timeout error.",
    "What are your support operating hours?",
]

for q in test_queries:
    out = support_graph.invoke({"query": q})
    print(f"Query: '{out['query']}' -> Category: {out['category']}")
    print(f"Response: {out['response']}\n")

Query: 'I have a billing issue with my invoice.' -> Category: billing
Response: [Billing Dept]: Processing refund/invoice request for 'I have a billing issue with my invoice.'.

Query: 'The API server returned a 504 timeout error.' -> Category: technical
Response: [Tech Support]: Investigating system logs for error in 'The API server returned a 504 timeout error.'.

Query: 'What are your support operating hours?' -> Category: general
Response: [Customer Care]: Assisting with inquiry: 'What are your support operating hours?'.


---
## 2. Pattern 2: Semantic Intent Router with LLM

Keyword rules fail when user queries are subtle, metaphorical, or complex.
A **Semantic Router** uses an LLM to understand the *underlying intent* of the prompt and dispatch it to dedicated expert nodes:
- `math_specialist`: Solves algebra, calculus, and arithmetic.
- `coding_specialist`: Generates and reviews code.
- `general_specialist`: Answers general trivia and conceptual questions.

In [3]:
class ExpertState(TypedDict):
    user_query: str
    destination: str
    final_answer: str

# 1. LLM Classifier Node: Determines user intent
def classifier_node(state: ExpertState):
    prompt = (
        "You are an intent router. Classify the user query into exactly one category: 'math', 'coding', or 'general'.\n"
        f"Query: {state['user_query']}\n"
        "Reply with ONLY the single category name in lowercase."
    )
    res = free_model.invoke(prompt)
    category = res.content.strip().lower()
    if "math" in category:
        dest = "math"
    elif "cod" in category or "python" in category:
        dest = "coding"
    else:
        dest = "general"
    return {"destination": dest}

# 2. Router Edge Function
def router_intent(state: ExpertState) -> str:
    return state["destination"]

# 3. Domain Expert Nodes
def math_specialist(state: ExpertState):
    res = free_model.invoke(f"You are a mathematics professor. Solve clearly step-by-step: {state['user_query']}")
    return {"final_answer": res.content}

def coding_specialist(state: ExpertState):
    res = free_model.invoke(f"You are a principal Python engineer. Provide concise, clean code: {state['user_query']}")
    return {"final_answer": res.content}

def general_specialist(state: ExpertState):
    res = free_model.invoke(f"You are a general knowledge expert. Answer succinctly: {state['user_query']}")
    return {"final_answer": res.content}

# 4. Graph Construction
expert_builder = StateGraph(ExpertState)
expert_builder.add_node("classifier", classifier_node)
expert_builder.add_node("math", math_specialist)
expert_builder.add_node("coding", coding_specialist)
expert_builder.add_node("general", general_specialist)

expert_builder.add_edge(START, "classifier")

# Routing to the chosen specialist
expert_builder.add_conditional_edges(
    "classifier",
    router_intent,
    {
        "math": "math",
        "coding": "coding",
        "general": "general",
    }
)

expert_builder.add_edge("math", END)
expert_builder.add_edge("coding", END)
expert_builder.add_edge("general", END)

expert_graph = expert_builder.compile()

# Graph Visualization for expert_graph
from IPython.display import Image, display
try:
    display(Image(expert_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(expert_graph.get_graph().draw_ascii())

# 5. Execute queries through semantic router
queries = [
    "Calculate the square root of 144 plus 25.",
    "Write a python function to check if a word is a palindrome.",
    "What is the capital of Japan and what is it famous for?",
]

for q in queries:
    result = expert_graph.invoke({"user_query": q})
    print("-" * 50)
    print(f"Query: '{q}'")
    print(f"Routed To: [{result['destination']}]")
    print(f"Specialist Answer:\n{result['final_answer']}")

--------------------------------------------------
Query: 'Calculate the square root of 144 plus 25.'
Routed To: [math]
Specialist Answer:
1. The square root of 144 is 12.
2. 12 + 25 = 37.
The final answer is 37.
--------------------------------------------------
Query: 'Write a python function to check if a word is a palindrome.'
Routed To: [coding]
Specialist Answer:
```python
def is_palindrome(word: str) -> bool:
    cleaned = word.lower().replace(' ', '')
    return cleaned == cleaned[::-1]
```
--------------------------------------------------
Query: 'What is the capital of Japan and what is it famous for?'
Routed To: [general]
Specialist Answer:
The capital of Japan is Tokyo. It is famous for its bustling modern skyline, historic shrines, world-class culinary scene, and vibrant technology and pop culture.


---
## 3. Pattern 3: Tool-Calling Router (`tools_condition`)

The most widely used router in agentic AI is the **Tool Router**.

When an agent node runs:
- If the LLM generates a `tool_calls` payload -> router sends execution to the `tools` node.
- If the LLM produces a regular text response (no tool calls) -> router terminates the loop and routes to `END`.

LangGraph provides `tools_condition` as a prebuilt conditional edge function specifically for this pattern.

In [4]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# 1. Define Tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    print(f"Executing multiply tool: {a} * {b} = {a * b}")
    return a * b

tools = [multiply]
model_with_tools = free_model.bind_tools(tools)

# 2. Agent Node
def agent_node(state: MessagesState):
    res = model_with_tools.invoke(state["messages"])
    return {"messages": [res]}

# 3. Graph Construction with tools_condition Router
tool_builder = StateGraph(MessagesState)
tool_builder.add_node("agent", agent_node)
tool_builder.add_node("tools", ToolNode(tools))

tool_builder.add_edge(START, "agent")

# Conditional edge: tools_condition routes to 'tools' or END
tool_builder.add_conditional_edges("agent", tools_condition)

# After tool executes, loop back to the agent to summarize
tool_builder.add_edge("tools", "agent")

tool_agent = tool_builder.compile()

# Graph Visualization for tool_agent
from IPython.display import Image, display
try:
    display(Image(tool_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(tool_agent.get_graph().draw_ascii())

# Test 1: Query requiring tool
print("--- Test 1: Query requiring tool execution ---")
r1 = tool_agent.invoke({"messages": [HumanMessage(content="What is 15 multiplied by 8?")]})
print("Final Answer:", r1["messages"][-1].content)

# Test 2: Query needing direct answer (no tools)
print("\n--- Test 2: Query requiring direct conversation (no tools) ---")
r2 = tool_agent.invoke({"messages": [HumanMessage(content="Hi! How are you?")]})
print("Final Answer:", r2["messages"][-1].content)

--- Test 1: Query requiring tool execution ---
Executing multiply tool: 15 * 8 = 120
Final Answer: 15 * 8 is **120**.

--- Test 2: Query requiring direct conversation (no tools) ---
Final Answer: Hello! I'm doing well, thank you. How can I help you today?


---
## 4. Routing Strategy Comparison Matrix

| Router Type | Mechanism | Latency | Token Cost | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Deterministic Rule Router** | Python `if/else`, regex, or state flags | **< 1 ms** | **$0.00** | Strict logic, ticket triage, high-volume fixed workflows |
| **Semantic LLM Intent Router** | LLM prompt classification / structured output | **200 - 500 ms** | **Low** | Ambiguous user queries, natural language multi-agent dispatch |
| **Tool Router (`tools_condition`)** | Checks `tool_calls` presence in `AIMessage` | **< 1 ms** | **$0.00** | Autonomous ReAct loops, deciding between tool execution vs. response |